<a href="https://colab.research.google.com/github/labgeo-poli/PTR3311-Python/blob/master/labs/lab4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laboratório 4

<br>

## Mosaicos e classificação supervisionada de imagens no GEE

<br>

**Docente responsável:** Prof.ª Dr.ª Mariana Abrantes Giannotti

**Elaboração do roteiro:** Guilherme Fernandes Alves - [Git](https://github.com/Gui-FernandesBR)

**Revisão, atualização e mantenimento:** Leonardo Alves Godoy - [Git](https://github.com/lagodoy)

**Monitoria:** Sabrina Franco Honorato

Objetivos:

1. Remoção de nuvens
2. Composições e mosaicos de imagens
3. Classificação supervisionada no GEE


In [ ]:
import ee
import geemap

In [ ]:
# Ref: https://developers.google.com/earth-engine/apidocs/ee-authenticate
# Para inicializar a sessão para execução insira o id do projeto em ee.Initialize().
ee.Authenticate()
ee.Initialize(project='id_projeto')

## Preparar dados


### Filtrar coleção de imagens (nuvens, datas e região)


Dessa vez, vamos utilizar uma coleção de imagens ao invés de uma única imagem.
Isso garantirá maior robustez ao nosso modelo de classificação.

Além disso, neste laboratório, diferentemente do anterior, será essencial utilizar imagens sem nuvens.
Para eliminar as nuvens, capturam-se múltiplas imagens da mesma área, removem-se as regiões cobertas por nuvens e, utilizando a técnica do mosaico, integram-se todas as imagens em uma única imagem.
Dessa forma, as lacunas deixadas pelas nuvens removidas são preenchidas, pois os pixels ausentes em uma imagem são substituídos por pixels válidos provenientes de outras imagens.

No Google Earth Engine (GEE), o pixel da imagem que está no topo da coleção tem prioridade na composição do mosaico.

Vamos ver como fazer isso.


In [ ]:
# Essa é uma função padrão do
def mask_s2_clouds(image):
    """Masks clouds in a Sentinel-2 image using the QA band.

    Args:
        image (ee.Image): A Sentinel-2 image.

    Returns:
        ee.Image: A cloud-masked Sentinel-2 image.
    """
    qa = image.select("QA60")

    # Bits 10 and 11 are clouds and cirrus, respectively.
    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11

    # Both flags should be set to zero, indicating clear conditions.
    mask = qa.bitwiseAnd(cloud_bit_mask).eq(0).And(qa.bitwiseAnd(cirrus_bit_mask).eq(0))

    return image.updateMask(mask).divide(10000)

Para nos auxiliar na análise, vamos filtrar apenas as imagens que abrangem a região de São Paulo, mais precisamente a área onde será desenhado um retângulo para recortar a imagem a ser classificada. Para isso, definiremos as coordenadas dos limites do polígono que delimita a área de interesse e utilizaremos o método `filterBounds()` do GEE para filtrar a imagem.


> Vamos definir os limites para cobertura de nuvem e filtrar a coleção de imagens.
Agora vamos filtrar o período em que queremos trabalhar com as imagens.

>Alguns comentários:

>- Selecionar um período muito longo pode resultar em uma coleção muito grande e pesada para trabalhar
>- Para fins de exemplos vamos cobrir o ano de 2020

>💡 A dica aqui é trabalhar com o intervalo de datas e valor de cobertura por
nuvens até se obter uma coleção de tamanho suficiente para a composição da imagem de forma apropriada








### Recortar a região de interesse (_Region of Interest - ROI_)


Primeiramente, vamos definir um polígono de interesse para a nossa análise.

Para fins de exemplo, vamos definir programaticamente um polígono retangular
que cobre a cidade de São Paulo, **com base nas coordenadas do retângulo criada no ambiente do GEE**.



In [ ]:
# roi: região de interesse
roi = ee.Geometry.BBox(
    west=-46.7669, south=-23.7397, east=-46.6814, north=-23.6361
)
roi

In [ ]:
limite_cobertura_nuvem = 5  # máximo de 10% de cobertura de nuvem
data_inicial, data_final = "2020-01-01", "2022-01-30"

dataset = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(roi)
    .filterDate(data_inicial, data_final)
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", limite_cobertura_nuvem))
    .map(mask_s2_clouds)
)
dataset

Podemos verificar o número de imagens resultantes na coleção com o método `size()`:


In [ ]:
print(f"total de imagens na coleção: {dataset.size().getInfo()}")

Como a ideia é utilizar uma coleção de imagens, o método `clip()`
não pode ser aplicado diretamente.

Para realizar esse procedimento, é necessário criar uma função personalizada.
No caso abaixo, criamos a função `clip_img_collection()`, que recebe uma
coleção de imagens e uma geometria (no caso, o nosso polígono de interesse)
e retorna a coleção já recortada.


In [ ]:
def clip_img_collection(dataset, geometria):
    """Recorta todas as imagens de uma coleção de imagens do Earth Engine para
    uma região específica definida por uma geometria.

    Essa função cria internamente uma função de recorte específica para a
    geometria fornecida e aplica essa função a cada imagem na coleção de imagens
    usando o método map().

    Parameters
    ----------
    dataset : ee.ImageCollection
        A coleção de imagens a ser recortada. Cada imagem da coleção será
        recortada para se adequar à geometria fornecida.

    geometria : ee.Geometry
        A geometria que define a região de interesse para onde as imagens serão
        recortadas. A geometria pode ser um ponto, linha, polígono, etc.

    Returns
    -------
    ee.ImageCollection
        Uma nova coleção de imagens com cada imagem recortada para a região
        definida pela geometria fornecida.
    """

    def clip_func(image):
        return image.clip(geometria)

    return dataset.map(clip_func)

In [ ]:
# Faz o clip da região desejada em toda coleção.
clipped_dataset = clip_img_collection(dataset, roi)
clipped_dataset

### Remoção de valores inválidos


Após a remoção das nuvens, vamos eliminar valores discrepantes para melhorar o conjunto de dados.

Sabemos que a reflectância deve ser menor do que 1. Portanto, vamos criar uma máscara para eliminar os valores superiores a 1.

As máscaras são geradas por meio do comparador `lt()` (_less than_ - menor do que) do GEE.


In [ ]:
# Função para remoção de valores inválidos
def remove_valores_invalidos(collection, bandas):
    """Aplica uma máscara para remover valores inválidos em uma coleção de
    imagens.

    Os valores considerados inválidos são aqueles maiores que 1 para as bandas
    especificadas.

    Parameters
    ----------
    collection : ee.ImageCollection
        A coleção de imagens a ser processada.
    bandas : list of str
        Lista com os nomes das bandas que serão processadas.

    Returns
    -------
    ee.ImageCollection
        A coleção de imagens com valores inválidos removidos.
    """

    def mask_out_of_range_reflectance(imagem):
        """Mascara valores de reflectância acima de 1"""
        for banda in bandas:
            imagem = imagem.updateMask(imagem.select(banda).lt(1))
        return imagem

    return collection.map(mask_out_of_range_reflectance)

In [ ]:
# Executa a função de remoção dos valores inválidos na coleção.
clipped_dataset = remove_valores_invalidos(clipped_dataset, ["B2", "B3", "B4", "B8"])
clipped_dataset

### Composição e Mosaico


Para criar um mosaico a partir de uma coleção de imagens, basta invocar o
método `mosaic()` da coleção de interesse.

Já para criar uma composição, deve-se chamar, a partir do objeto que contém a coleção, um método do tipo reducer, que realiza um cálculo pixel a pixel. Nesse caso, os pixels transparentes são ignorados durante os cálculos.

Os cálculos são realizados separadamente para cada banda das imagens da coleção,
resultando em uma imagem com as mesmas bandas da coleção. Nessa imagem, cada pixel de cada banda é preenchido por um valor calculado a partir da banda de mesmo nome presente na coleção.


In [ ]:
# Criação de mosaico a partir da coleção.
#mosaico = clipped_dataset.mosaic()


# Criação de composição a partir da coleção.
composition = clipped_dataset.mean()

Vamos conferir o resultado da composição através do método `bandTypes()` que
retorna as bandas da imagem.


In [ ]:
composition.bandTypes()
# TODO: for a strange reason, it is giving me numbers higher than 1.0, needs checking

### Calcular NDVI


Agora queremos calcular a banda de índice de vegetação (`NDVI`) para cada imagem
da coleção.
Vamos seguir os mesmos passos do laboratório anterior!

Alguns comentários:

- `B2` é a banda azul
- `B3` é a banda verde
- `B4` é a banda vermelha
- `B8` é a banda infravermelho próximo (NIR)
- `MSK_CLDPRB` é a banda de probabilidade de nuvens
- `SCL` é fruto de um processo de classificação pelo algoritmo de classificação de cena ([_Scene Classification_](https://earth.esa.int/web/sentinel/technical-guides/sentinel-2-msi/level-2a/algorithm))

💡 Sempre verifique a documentação oficial: https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED#bands


In [ ]:
# calcular a banda NDVI conforme visto no laboratório anterior.

img_bd_nir = composition.select("B8")
img_bd_red = composition.select("B4")

## adicionar a banda NDVI à coleção de imagens
composition = composition.addBands(
    img_bd_nir.subtract(img_bd_red).divide(img_bd_nir.add(img_bd_red)).rename("NDVI"),
    ["NDVI"],
)
composition

### Seleção de bandas para a classificação


Vamos criar listas com nomes das bandas para utilizarmos ao longo deste laboratório.


In [ ]:
# Lista com somente as bandas de interesse para a classificação
classification_bands = ["B2", "B3", "B4", "B8", "NDVI"]

In [ ]:
# Descarta as bandas usadas para se eliminar efeitos de nuvens.
composition = composition.select(classification_bands)
composition

###Salvando um arquivo geotiff no Drive

Após a conclusão do processamento abaixo, clique no ícone em formato de pasta na barra de ferramentas localizada à esquerda e localize o arquivo "sentinel.tif". Para baixá-lo, clique nos três pontos à direita do nome do arquivo e selecione a opção download.

In [ ]:
import os
arquivo_saida='sentinel.tif'
print(arquivo_saida)
imagem_final = composition.clip(roi).unmask()
geemap.ee_export_image(
    imagem_final, filename=arquivo_saida, scale=10, region=roi, file_per_band=False
)

## Classificação supervisionada


### Definindo amostras conhecidas


Vamos começar a definir visualmente regiões conhecidas para treinamento do
modelo de classificação.
Em outras palavras, precisamos gerar amostras sobre a imagem para que o modelo
aprenda a classificar os pixels.




Vamos utilizar o QGIS que permite a criação de um *layer* vetorial com polígonos com atributos sobre uma camada com imagem para referência.

Para a criação do arquivo de amostras, você vai precisar de um campo onde cada polígono vai reber um código em número inteiro para de acordo com a classe que ele representa, iniciando a partir de 0 e seguindo somando mais 1 para cada nova classe. Essa lógica é importante pela representação vetorial (aqui não no sentido de dados vetoriais em SIG!) dos dados no algoritmo de classificação. Orientações no roteiro disponibilizado.


Após a criação do arquivo vetorial, você deve fazer o upload dos dados para o ambiente do *colab*.

O código a seguir, habilita o acesso aos arquivos de seu Google Drive. Caso alguma janela apareça, dê autorização para o colab acessar seus arquivos.

In [ ]:
# Conecta ao drive.
from google.colab import drive
drive.mount('/content/drive')

Agora vamos instalar o pacote pycrs, utilizado para conversão entre sistemas de referência de coordenadas.

In [ ]:
# Instala o pacote pycrs.
!pip install pycrs
!pip install fiona

No box abaixo, as amostras a partir de um arquivo shp no Gogole Drive são importadas. É necessário ajustar o caminho da pasta e nome do arquivo com suas amostras.

In [ ]:
#amostras_completas = geemap.shp_to_ee("/content/drive/MyDrive/amostras/amostras.shp")
#amostras_completas

data = "/content/drive/MyDrive/amostras/amostras.gpkg"
amostras_completas=geemap.vector_to_ee(data)
amostras_completas

Vamos definir um dicionário para armazenar os parâmetros de visualização básicos para o mapa.


In [ ]:
visualization = {
    "min": 0.0,
    "max": 0.3,
    "bands": ["B4", "B3", "B2"],
}

### Sampling


O procedimento anterior apenas delimitou as regiões de onde se deseja coletar
amostra, associando-as com suas respectivas classes, e não colheu nenhuma
amostra propriamente.

Vamos fazer a coleta dos valores de pixels nas regiões.


Adicionalmente, vamos definir a escala que queremos trabalhar, nesse caso 10 metros pois é o
menor valor (mais preciso) disponível para o Sentinel-2.


In [ ]:
# Ajuste este valor para aumentar ou diminuir a área de amostragem
SCALE = 10  # meters

In [ ]:
# Extrai o valor de todos os pixels nas regiões amostradas.
amostra_treinamento = composition.sampleRegions(
    collection= amostras_completas, properties=["classe"], scale=SCALE
)

In [ ]:
amostra_treinamento

### Treinamento


Para treinar o modelo, basta utilizar um classificador com os parâmetros desejados
e utilizar o método `train()`.

Vamos utilizar o classificador [Random Forest](https://developers.google.com/earth-engine/apidocs/ee-classifier-smilerandomforest).

O primeiro parâmetro do `train()` é a amostra a ser utilizada para treinar o modelo, o segundo é a propriedade que contém o número que identifica a classe e o último, a lista com as bandas que serão utilizadas para classificação.

Na instanciação do classificador, `smileRandomForest()`, o parâmetro é a quantidade de árvores de decisão que devem ser criadas.
Esse valor pode, e deve, ser ajustado de acordo com os resultados no conjunto de validação.

Outros parâmetros podem ser encontrados na documentação do GEE e a explicação da utilidade dos mesmos em literatura apropriada.


In [ ]:
# Instancia um classificador na memória com os parâmetros dados e treinando no conjunto de treinamento.
classificador_treinado = ee.Classifier.smileRandomForest(50).train(
    amostra_treinamento, "classe", classification_bands
)
classificador_treinado

Um primeiro teste pode ser feito com a própria amostra de treinamento, com a criação da matriz de confusão a partir do método `confusionMatrix()`.

A acurácia provavelmente será altíssima, visto que foi o próprio conjunto utilizado no treinamento que foi avaliado.

Caso o desempenho tenha sido ruim sobre o conjunto de treinamento, é um provável caso de [underfitting](https://en.wikipedia.org/wiki/Overfitting).


In [ ]:
# // Matriz de confusão do conjunto de treinamento e acurácia.
matriz_confusao = classificador_treinado.confusionMatrix()
matriz_confusao

In [ ]:
print(
    f"Acurácia no conjunto de treinamento: {matriz_confusao.accuracy().getInfo()*100:.4f}%"
)

### Classificação


Para classificar a imagem a partir do treinamento, invoca-se o método `classify()` a partir da imagem e com o classificador treinado escolhido como parâmetro.

O resultado é um objeto do tipo imagem com uma única banda em que os pixels
armazenam o valor relativo às classes que foram atribuídas no momento do desenho
das regiões das amostras em tela.


In [ ]:
# Classifica a imagem com o classificador treinado e com os parâmetros definidos.
imagem_classificada = composition.classify(classificador_treinado)
imagem_classificada

### Visualização


Para visualizar a imagem, primeiro uma paleta de cores deve ser definida.

Essa paleta deve ser feita como uma lista de strings em que cada cor é uma sequência de seis dígitos em hexadecimal.

A ordem em que as cores estão na string representa o número inteiro associado com a classe a partir de 0.


In [ ]:
# Paleta de cores para adicionar a imagem classificada ao mapa.
paleta_cores_classes = [
    "#2915d6",  # Água (0) - um azul mais suave e claro.
    "#08a910",  # Vegetação alta (1) - um verde mais vibrante e menos saturado.
    "#e17f04",  # Solo exposto (2) - um laranja mais vibrante e atraente.
    "#14fb3b",  # Vegetação rasteira (3) - um verde claro para diferenciar da vegetação alta.
    "#aaaaaa",  # Construção(4) - um cinza médio que representa áreas construídas.
]

Finalmente, podemos visualizar a imagem classificada.


In [ ]:
final_map = geemap.Map(lite_mode=False)
final_map.centerObject(roi, 12)
vis_params = {
    "min": 0,
    "max": 4,
    "bands": ["classification"],
    "palette": paleta_cores_classes,
}
final_map.addLayer(
    composition, visualization, "Imagem", True
)
final_map.addLayer(
    imagem_classificada.clip(roi), vis_params, "Imagem Classificada", True
)
final_map

## Exportar a imagem classificada para um shapefile


O código a seguir converte o raster da classificação para vetor, agregando pixels idênticos contíguos. Para cada polígono determinado, será calculado o respectivo total de pixels, permitindo assim a totalização da área associada.


In [ ]:
classificacao_vetorizada = imagem_classificada.reduceToVectors(**{
  'geometry': roi,
  'crs': imagem_classificada.projection(),
  'scale': SCALE,
  'geometryType': 'polygon',
  'eightConnected': False,
  'labelProperty': 'classe'#,
  #'reducer': ee.Reducer.mean()
});
classificacao_vetorizada = classificacao_vetorizada.filter(ee.Filter.gte('count',10))
#display = ee.Image(0).updateMask(0).paint(classificacao_vetorizada, '000000', 3)
#final_map.addLayer(display, {'palette': '000000'}, 'Classificação')
#final_map


Salva o arquivo vetorial gerado em seu drive. Não esqueça de criar a pasta para receber o shapefile e ajustar o caminho e nome no parâmetro filename.

In [ ]:
geemap.ee_to_shp(classificacao_vetorizada, filename="/content/drive/MyDrive/vetorizado/vetorizado.shp")